In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.cloud import storage
import io

In [2]:
import google.auth
from google.auth import impersonated_credentials

credentials, project = google.auth.default()
print(credentials, project)

service_account = "le-wagon-bootcamp@le-wagon-2303.iam.gserviceaccount.com"
signing_credentials = impersonated_credentials.Credentials(
    source_credentials=credentials,
    target_principal=service_account,
    target_scopes=["https://www.googleapis.com/auth/cloud-platform"],
    lifetime=3600,
)

bucket_client = storage.Client(project='le-wagon-2303',  credentials=signing_credentials)
bucket = bucket_client.bucket('sm-optimizer-processed')

clean_blob = bucket.blob("posts_clean.csv")
df = pd.read_csv(clean_blob.open('rb'))
df.head()

<google.oauth2.service_account.Credentials object at 0xfffead6b19d0> wagon-ds-2026


,campaign,year,page,platform,media_type,category_l0,category_l1,category_l2,url,content,cost_nzd,views,hours,engagement
0,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/729678446160620/,@jacobkneepkens crosses the chalk 🤙,NaN,38409.0,64.3328,682.0
1,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/850774540642042/,@nanaiseturo 💨,NaN,235384.0,544.8062,3800.0
2,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1120082556960744/,a classic rugby training drill 🏉,NaN,274028.0,712.1043,3261.0
3,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1191989952906685/,"a great kick, a great backdrop 🎨",NaN,281319.0,623.9321,4929.0
4,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1728500341153571/,a skill and a half 🚀,NaN,438181.0,894.3045,9850.0


In [3]:
### existing descriptions JSONs
yt_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/youtube")
yt_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in yt_description_blobs]

tt_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/tiktok")
tt_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in tt_description_blobs]

fb_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/facebook")
fb_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in fb_description_blobs]

ig_description_blobs = bucket_client.list_blobs('sm-optimizer-processed', prefix="gemini-descriptions/instagram")
ig_description_keys = [f"{blob.name.split('/')[-1].removesuffix('.json')}" for blob in ig_description_blobs]

all_description_keys = yt_description_keys + tt_description_keys + fb_description_keys + ig_description_keys
print(all_description_keys[12], all_description_keys[-40])
print('descriptions in bucket:', len([k for k in all_description_keys if k]))

-D1tJ_VPX9A DVsUnHNE7MK
descriptions in bucket: 12601


In [4]:
import json
import re
from concurrent.futures import ThreadPoolExecutor

PLATFORM_TO_PREFIX = {"YT": "yt", "TT": "tt", "FB": "fb", "IG": "ig"}
PLATFORM_TO_FOLDER = {"YT": "youtube", "TT": "tiktok", "FB": "facebook", "IG": "instagram"}

URL_ID_PATTERNS = {
    "FB": r"/(?:reel|video|videos|posts|share|v)/([\w-]+)",
    "YT": r"(?:v=|youtu\.be/|embed/)([\w-]{6,})",
    "TT": r"/video/(\d+)",
    "IG": r"/(?:p|reel|tv)/([\w-]+)",
}

def _url_id(url, platform):
    if pd.isna(url):
        return None
    match = re.search(URL_ID_PATTERNS[platform], str(url))
    return match.group(1) if match else None

platform_description_keys = {
    "YT": set(yt_description_keys),
    "TT": set(tt_description_keys),
    "FB": set(fb_description_keys),
    "IG": set(ig_description_keys),
}
key_to_blob = {
    (platform, url_id): f"gemini-descriptions/{PLATFORM_TO_FOLDER[platform]}/{url_id}.json"
    for platform, keys in platform_description_keys.items()
    for url_id in keys
}

platforms = df["platform"].tolist()
urls = df["url"].tolist()
url_ids = [_url_id(url, plat) for url, plat in zip(urls, platforms)]

valid_pairs = [
    (plat, uid) if plat in PLATFORM_TO_PREFIX and uid is not None and (plat, uid) in key_to_blob else None
    for plat, uid in zip(platforms, url_ids)
]
matched_pos = [i for i, p in enumerate(valid_pairs) if p is not None]
matched_indices = df.index[matched_pos]
blob_paths = [key_to_blob[valid_pairs[i]] for i in matched_pos]

def _fetch_raw(blob_path):
    return bucket.blob(blob_path).download_as_bytes().decode("utf-8")

with ThreadPoolExecutor(max_workers=32) as ex:
    json_strings = list(ex.map(_fetch_raw, blob_paths))

df = df.loc[matched_indices].copy()
df["description_json"] = pd.Series(json_strings, index=matched_indices, dtype="string")

print(f"Good matches: {len(json_strings)}, unmatched bucket keys: {len([k for k in all_description_keys if k]) - len(json_strings)}")


Good matches: 12601, unmatched bucket keys: 0


In [11]:
df = df[df['description_json'].notna()]
df = df.dropna(subset=["views", "engagement"], how="any")
df = df[df['engagement'] >= 0]
df = df[df['views'] >= 0]
print(len(df))
df.head()


12105


,campaign,year,page,platform,media_type,category_l0,category_l1,category_l2,url,content,cost_nzd,views,hours,engagement,description_json,duration_seconds
0,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/729678446160620/,@jacobkneepkens crosses the chalk 🤙,NaN,38409.0,64.3328,682.0,"{""play_by_play"": ""A male rugby player in a bla...",<NA>
1,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/850774540642042/,@nanaiseturo 💨,NaN,235384.0,544.8062,3800.0,"{""play_by_play"": ""A male rugby player wearing ...",<NA>
2,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1120082556960744/,a classic rugby training drill 🏉,NaN,274028.0,712.1043,3261.0,"{""play_by_play"": ""A group of male rugby player...",<NA>
3,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1191989952906685/,"a great kick, a great backdrop 🎨",NaN,281319.0,623.9321,4929.0,"{""play_by_play"": ""A male rugby player runs up ...",<NA>
4,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1728500341153571/,a skill and a half 🚀,NaN,438181.0,894.3045,9850.0,"{""play_by_play"": ""A male All Black player trac...",<NA>


In [13]:
from io import BytesIO

from mutagen.mp4 import MP4
from concurrent.futures import ThreadPoolExecutor

MP4_HEADER_BYTES = 1 * 1024 * 1024


def _mp4_duration_seconds(blob):
    try:
        head = blob.download_as_bytes(start=0, end=MP4_HEADER_BYTES - 1)
        if not head:
            return None
        return MP4(BytesIO(head)).info.length
    except Exception:
        return None


def _video_blob_path(platform, url_id):
    return f"videos/{PLATFORM_TO_FOLDER[platform]}/{url_id}.mp4"


platforms_now = df["platform"].tolist()
url_ids_now = [_url_id(url, plat) for url, plat in zip(df["url"].tolist(), platforms_now)]


def _duration_for_row(i):
    plat = platforms_now[i]
    uid = url_ids_now[i]
    if plat not in PLATFORM_TO_PREFIX or uid is None:
        return None
    return _mp4_duration_seconds(bucket.blob(_video_blob_path(plat, uid)))


with ThreadPoolExecutor(max_workers=16) as ex:
    durations = list(ex.map(_duration_for_row, range(len(df))))

df["duration_seconds"] = pd.Series(durations, index=df.index, dtype="Float64")

parsed = sum(d is not None for d in durations)
print(f"Parsed durations: {parsed} / {len(durations)} (failures: {len(durations) - parsed})")


Parsed durations: 12105 / 12105 (failures: 0)


In [14]:
LOCAL_CSV_PATH = "processed.csv"

df.to_csv(LOCAL_CSV_PATH, index=False)

print(f"Saved {len(df)} rows to {LOCAL_CSV_PATH}")


Saved 12105 rows to processed.csv


In [18]:
df.head()

,campaign,year,page,platform,media_type,category_l0,category_l1,category_l2,url,content,cost_nzd,views,hours,engagement,description_json,duration_seconds
0,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/729678446160620/,@jacobkneepkens crosses the chalk 🤙,NaN,38409.0,64.3328,682.0,"{""play_by_play"": ""A male rugby player in a bla...",12.839229
1,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/850774540642042/,@nanaiseturo 💨,NaN,235384.0,544.8062,3800.0,"{""play_by_play"": ""A male rugby player wearing ...",14.279229
2,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1120082556960744/,a classic rugby training drill 🏉,NaN,274028.0,712.1043,3261.0,"{""play_by_play"": ""A group of male rugby player...",19.644082
3,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1191989952906685/,"a great kick, a great backdrop 🎨",NaN,281319.0,623.9321,4929.0,"{""play_by_play"": ""A male rugby player runs up ...",15.625215
4,Organic | Website,2025,ABXV,FB,Short Video,No Hashtag,No Hashtag,No Hashtag,https://www.facebook.com/reel/1728500341153571/,a skill and a half 🚀,NaN,438181.0,894.3045,9850.0,"{""play_by_play"": ""A male All Black player trac...",13.605215
